# einops-reduce-min — worked example 1: Compute the minimum over channels for each spatial position

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-reduce-min`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`einops.reduce` with `'min'` collapses any axis named on the left but missing from the right by taking the element-wise minimum. The pattern `'b c h w -> b h w'` drops the channel axis by taking the min across all channels at each spatial location. This gives the spatially-resolved channel minimum — useful for computing background estimates or detecting where any channel activates.

## Worked solution

Input: `(B=2, C=5, H=4, W=4)` activation tensor.

**Pattern:** `'b c h w -> b h w'` with `'min'`.

For each `(b, h, w)` location, we take the minimum across all 5 channels: `out[b, h, w] = min_c x[b, c, h, w]`.

**Result shape:** `(2, 4, 4)`. Each spatial position now holds the smallest channel value at that location.

This is the dual of `'b c h w -> b c'` (spatial min): here we reduce over channels to get a spatial map; there we reduce over space to get a per-channel number.

In [ ]:
import torch as t
from einops import reduce

t.manual_seed(8)
B, C, H, W = 2, 6, 5, 5
x = t.randn(B, C, H, W)

def channel_min_map(x):
    """For each (b, h, w), take the min over channels."""
    return reduce(x, 'b c h w -> b h w', 'min')

out = channel_min_map(x)
print('Input shape:', x.shape)
print('Output shape:', out.shape)  # (2, 5, 5)
assert out.shape == (B, H, W)

# Verify against torch
expected = x.min(dim=1).values
assert t.allclose(out, expected)
print('Matches x.min(dim=1).values:', True)